# Dataset Preprocessing

Transform the raw signals (EEG, ET, HR) into a clean dataset, ready to use for modeling.

**Pipeline:**
1. Eye Tracking: Extract 4 key features and calculate mean/std per meme.
2. EEG: z-score per subject, then features per channel and band, then extra features (differences, ratios, globals)
3. Expansion of Task21, Task22, Task23.


**Output:**
- Train: `data/processed/train_model_ready.parquet`
- Test: `data/processed/test_model_ready.parquet`


In [3]:
#Libraries
import os
import ast
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
tqdm.pandas()


#os.chdir("C:/Users/diego/Desktop/Master Neuro/M2/Internship_Valencia/multimodal-exist")
os.chdir("/home/diegoz/projects/multimodal-exist")

In [75]:
#Paths
OUTPUT_DIR = os.path.join("data", "processed")
os.makedirs(OUTPUT_DIR, exist_ok=True)

#Load Dataset
train_file = os.path.join(OUTPUT_DIR, "train_base.parquet")
test_file = os.path.join(OUTPUT_DIR, "test_base.parquet")

df_train = pd.read_parquet(train_file)
#df_test = pd.read_parquet(test_file)

print(f"  Train: {df_train.shape[0]} samples")
#print(f"  Test:  {df_test.shape[0]} samples")

  Train: 3984 samples


In [76]:
df_train.head()

,id,lang,text,image_file,split,ET_raw,HR_raw,EEG_raw,task21_hard,task21_valid_hard,task21_soft,task22_hard,task22_valid_hard,task22_soft,task23_hard,task23_valid_hard,task23_soft,task23_num_hard_labels
0,110887,es,A VECES QUISIERAIR/ALZUMBA www.facebook.com/Oi...,110887.jpeg,Train,"{'EN1': None, 'EN2': None, 'EN3': None, 'EN4':...","{'EN1': None, 'EN2': None, 'EN3': None, 'EN4':...","{'EN1': None, 'EN2': None, 'EN3': None, 'EN5':...",1,True,1.000000,1.0,True,0.833333,"{'IDEOLOGICAL-INEQUALITY': 0, 'MISOGYNY-NON-SE...",True,{'IDEOLOGICAL-INEQUALITY': 0.16666666666666666...,2
1,110466,es,Se necesita cuidadora para adulto mayor.... fo...,110466.jpeg,Train,"{'EN1': None, 'EN2': None, 'EN3': None, 'EN4':...","{'EN1': None, 'EN2': None, 'EN3': None, 'EN4':...","{'EN1': None, 'EN2': None, 'EN3': None, 'EN5':...",1,True,0.500000,1.0,True,1.000000,"{'IDEOLOGICAL-INEQUALITY': 0, 'MISOGYNY-NON-SE...",True,{'IDEOLOGICAL-INEQUALITY': 0.16666666666666666...,2
2,111269,es,tomboy como son el anime y manga pre to tomboy...,111269.jpeg,Train,"{'EN1': None, 'EN2': None, 'EN3': None, 'EN4':...","{'EN1': None, 'EN2': None, 'EN3': None, 'EN4':...","{'EN1': None, 'EN2': None, 'EN3': None, 'EN5':...",1,True,0.833333,1.0,True,1.000000,"{'IDEOLOGICAL-INEQUALITY': 0, 'MISOGYNY-NON-SE...",True,"{'IDEOLOGICAL-INEQUALITY': 0.0, 'MISOGYNY-NON-...",1
3,110593,es,HOY QUIERO FELICITAR A TODAS LAS MUJERES DE ES...,110593.jpeg,Train,"{'EN1': None, 'EN2': None, 'EN3': None, 'EN4':...","{'EN1': None, 'EN2': None, 'EN3': None, 'EN4':...","{'EN1': None, 'EN2': None, 'EN3': None, 'EN5':...",0,True,0.166667,0.0,True,0.000000,"{'IDEOLOGICAL-INEQUALITY': 0, 'MISOGYNY-NON-SE...",False,"{'IDEOLOGICAL-INEQUALITY': 0.0, 'MISOGYNY-NON-...",0
4,110946,es,DUCHATE GUARRA GRACIAS memegenerator.es,110946.jpeg,Train,"{'EN1': None, 'EN2': None, 'EN3': None, 'EN4':...","{'EN1': None, 'EN2': None, 'EN3': None, 'EN4':...","{'EN1': None, 'EN2': None, 'EN3': None, 'EN5':...",1,True,0.666667,1.0,True,1.000000,"{'IDEOLOGICAL-INEQUALITY': 0, 'MISOGYNY-NON-SE...",True,"{'IDEOLOGICAL-INEQUALITY': 0.0, 'MISOGYNY-NON-...",1


## 1. Eye Tracking Aggregation

Selected Features (based on cognitive relevance - Sensorial Analysis)
- `reaction_time` (ms to s).
- `fixations_count`.
- `fixations_duration_mean_ns` (ns to ms).
- `saccades_count`.

Per meme we extract the mean and the std of valid subjects.

In [77]:
ET_FEATURE_MAP = {
    "reaction_time":"et_reaction_time",
    "fixations_count":"et_fixations",
    "fixations_duration_mean_ns":"et_fixation_duration",
    "saccades_count":"et_saccades"}

In [78]:
def aggregate_et_features(et_raw):

    result = {"et_n_users": 0}
    for feat_name in ET_FEATURE_MAP.values():
        result[f"{feat_name}_mean"] = np.nan
        result[f"{feat_name}_std"]  = np.nan

    if not isinstance(et_raw, dict):
        return result

    valid_users = {u: f for u, f in et_raw.items() if isinstance(f, dict)}
    result["et_n_users"] = len(valid_users)

    if len(valid_users) == 0:
        return result


    collected = {k: [] for k in ET_FEATURE_MAP}
    for _, feats in valid_users.items():
        for raw_key in ET_FEATURE_MAP:
            v = feats.get(raw_key, None)
            if v is not None:
                collected[raw_key].append(v)

    collected["reaction_time"]             = [x / 1_000     for x in collected["reaction_time"]]
    collected["fixations_duration_mean_ns"] = [x / 1_000_000 for x in collected["fixations_duration_mean_ns"]]

    for raw_key, feat_name in ET_FEATURE_MAP.items():
        vals = pd.to_numeric(pd.Series(collected[raw_key]), errors="coerce").dropna()
        if len(vals) > 0:
            result[f"{feat_name}_mean"] = vals.mean()
            result[f"{feat_name}_std"]  = vals.std() if len(vals) > 1 else 0.0

    return result

In [79]:
for df, name in [(df_train, "train")]:
    et_agg = df["ET_raw"].apply(aggregate_et_features).apply(pd.Series)
    et_cols_to_add = [c for c in et_agg.columns if c not in df.columns]
    df[et_agg.columns] = et_agg.values

et_feature_cols = [c for c in df_train.columns if c.startswith("et_")]
print(f"\nET columns ({len(et_feature_cols)}): {et_feature_cols}")


ET columns (9): ['et_n_users', 'et_reaction_time_mean', 'et_reaction_time_std', 'et_fixations_mean', 'et_fixations_std', 'et_fixation_duration_mean', 'et_fixation_duration_std', 'et_saccades_mean', 'et_saccades_std']


## 2. EEG Aggregation

1. Z-score per subject
2. Raw Features per channel (16 channels * 5 bands = 80 raw)
3. Global Features (mean between channels)
4. Extra Features (Diferences ('`Theta − Alpha`,...), Ratio, )
5. We agregate per subject: mean and std per meme


In [80]:
# EEG Constants
BANDS = ["Alpha", "Beta", "Theta", "Gamma", "Delta"]
N_CH  = 16
AUX   = 1e-6   

#Mapping
REGION_MAP = {
    "frontal_polar": [0, 1],         # Fp1, Fp2
    "frontal":       [8, 9, 10, 11], # F7, F8, F3, F4
    "central":       [2, 3],         # C3, C4
    "temporal":      [12, 13],       # T7, T8
    "parietal":      [4, 5, 14, 15], # P7, P8, P3, P4
    "occipital":     [6, 7],         # O1, O2
}

LEFT_CHANNELS  = [0, 2, 4, 6, 8, 10, 12, 14]  # Fp1, C3, P7, O1, F7, F3, T7, P3
RIGHT_CHANNELS = [1, 3, 5, 7, 9, 11, 13, 15]  # Fp2, C4, P8, O2, F8, F4, T8, P4

#Feature that we are going to include base on our statictical analysis

#Log-Ratios 
LOG_RATIO_PAIRS = [
    ("Beta",  "Alpha"),
    ("Theta", "Beta"),
    ("Theta", "Alpha"),
    ("Gamma", "Beta"),
    ("Gamma", "Alpha")]

#Global Differences
GLOBAL_DIFF_PAIRS = [
    ("Alpha", "Delta"),   # Task 2.1, SEXUAL-VIOLENCE
    ("Alpha", "Theta"),   # Task 2.1, Task 2.2, SEXUAL-VIOLENCE
    ("Alpha", "Beta"),
    ("Beta",  "Gamma"),   # Task 2.1
    ("Beta",  "Delta"),
    ("Beta",  "Theta"),
    ("Delta", "Gamma"),
    ("Delta", "Theta"),   # SEXUAL-VIOLENCE
    ("Gamma", "Theta"),]

#Regional differences
REGIONAL_DIFF_PAIRS = [
    ("Alpha", "Theta",  "frontal_polar"),# frontal_polar — Task 2.2 y Task 2.3 IDEOLOGICAL-INEQUALITY
    ("Alpha", "Beta",   "frontal_polar"),
    ("Alpha", "Gamma",  "frontal_polar"),
    ("Beta",  "Delta",  "frontal_polar"),
    ("Beta",  "Theta",  "frontal_polar"),
    ("Delta", "Gamma",  "frontal_polar"),
    ("Gamma", "Theta",  "frontal_polar"),# central — Task 2.1 y Task 2.3 SEXUAL-VIOLENCE
    ("Alpha", "Gamma",  "central"),
    ("Alpha", "Theta",  "central"),
    ("Beta",  "Gamma",  "central"),
    ("Beta",  "Delta",  "central"),
    ("Delta", "Gamma",  "central"),
    ("Delta", "Theta",  "central"),# temporal — Task 2.3 SEXUAL-VIOLENCE
    ("Beta",  "Delta",  "temporal"),
    ("Delta", "Theta",  "temporal"),# parietal — Task 2.3 STEREOTYPING-DOMINANCE
    ("Alpha", "Beta",   "parietal"),# occipital — Task 2.3 STEREOTYPING-DOMINANCE
    ("Beta",  "Theta",  "occipital"),
    ("Gamma", "Theta",  "occipital"),# frontal — Task 2.2
    ("Alpha", "Theta",  "frontal"),
]

print(f"  Bands                       : {BANDS}")
print(f"  Regions                     : {list(REGION_MAP.keys())}")
print(f"  Log-ratio pairs             : {len(LOG_RATIO_PAIRS)} × 6 regions = {len(LOG_RATIO_PAIRS)*6}")
print(f"  Global diff pairs selected  : {len(GLOBAL_DIFF_PAIRS)}")
print(f"  Regional diff pairs selected: {len(REGIONAL_DIFF_PAIRS)}")

  Bands                       : ['Alpha', 'Beta', 'Theta', 'Gamma', 'Delta']
  Regions                     : ['frontal_polar', 'frontal', 'central', 'temporal', 'parietal', 'occipital']
  Log-ratio pairs             : 5 × 6 regions = 30
  Global diff pairs selected  : 9
  Regional diff pairs selected: 19


In [84]:
#Function to standardize the EEG features of each user (z-scoring)
def zscore_user_eeg(user_feats):
    keys = [f"EXG_Channel_{ch}_{band}_power"
            for ch in range(N_CH) for band in BANDS]
    vals = np.array([user_feats.get(k, np.nan) for k in keys], dtype=float)

    mask = ~np.isnan(vals)
    if mask.sum() > 1:
        mu, sigma = vals[mask].mean(), vals[mask].std()
        if sigma > 0:
            vals[mask] = (vals[mask] - mu) / sigma

    return dict(zip(keys, vals))


#Function to create features
def extract_eeg_user_features(d):

    feats = {}

    # Raw Band Power por canal (16 ch × 5 bandas = 80 features)
    for ch in range(N_CH):
        for band in BANDS:
            key = f"EXG_Channel_{ch}_{band}_power"
            feats[key] = d.get(key, np.nan)

    # For each band, we calculate the mean across all channels (5 features)
    for band in BANDS:
        vals = [d.get(f"EXG_Channel_{ch}_{band}_power", np.nan) for ch in range(N_CH)]
        vals = [v for v in vals if not np.isnan(v)]
        feats[f"EEG_{band}_global"] = np.mean(vals) if vals else np.nan

    #For each region, we calculate the mean acrross its channels.
    for region, ch_list in REGION_MAP.items():
        for band in BANDS:
            vals = [d.get(f"EXG_Channel_{ch}_{band}_power", np.nan) for ch in ch_list]
            vals = [v for v in vals if not np.isnan(v)]
            feats[f"EEG_{band}_{region}"] = np.mean(vals) if vals else np.nan

    #Laterilazation: For each band, we calculate the mean for left and right channels.
    for band in BANDS:
        l_vals = [d.get(f"EXG_Channel_{ch}_{band}_power", np.nan) for ch in LEFT_CHANNELS]
        r_vals = [d.get(f"EXG_Channel_{ch}_{band}_power", np.nan) for ch in RIGHT_CHANNELS]
        l_vals = [v for v in l_vals if not np.isnan(v)]
        r_vals = [v for v in r_vals if not np.isnan(v)]
        feats[f"EEG_{band}_left"]  = np.mean(l_vals) if l_vals else np.nan
        feats[f"EEG_{band}_right"] = np.mean(r_vals) if r_vals else np.nan

    #Global Differences
    for b1, b2 in GLOBAL_DIFF_PAIRS:
        v1 = feats.get(f"EEG_{b1}_global", np.nan)
        v2 = feats.get(f"EEG_{b2}_global", np.nan)
        feats[f"EEG_{b1}_minus_{b2}_global"] = (v1 - v2 if not (np.isnan(v1) or np.isnan(v2)) else np.nan)

    #Regional Differences
    for b1, b2, region in REGIONAL_DIFF_PAIRS:
        v1 = feats.get(f"EEG_{b1}_{region}", np.nan)
        v2 = feats.get(f"EEG_{b2}_{region}", np.nan)
        feats[f"EEG_{b1}_minus_{b2}_{region}"] = (v1 - v2 if not (np.isnan(v1) or np.isnan(v2)) else np.nan)

    #Log Ratios
    for region in REGION_MAP.keys():
        for num, den in LOG_RATIO_PAIRS:
            v_num = feats.get(f"EEG_{num}_{region}", np.nan)
            v_den = feats.get(f"EEG_{den}_{region}", np.nan)
            if not (np.isnan(v_num) or np.isnan(v_den)):
                feats[f"EEG_log_{num}_{den}_ratio_{region}"] = np.log((v_num + AUX) / (v_den + AUX))
            else:
                feats[f"EEG_log_{num}_{den}_ratio_{region}"] = np.nan

    return feats

# Renaming the features to be more clear and consistent 
_dummy_d = {f"EXG_Channel_{ch}_{band}_power": 0.0
            for ch in range(N_CH) for band in BANDS}
_dummy_feats    = extract_eeg_user_features(_dummy_d)
EEG_FEATURE_NAMES = list(_dummy_feats.keys())

print(f"  Total features por sujeto  : {len(EEG_FEATURE_NAMES)}")
print(f"  Raw por canal (16*5)        : 80")
print(f"  Globales (5 bandas)         : 5")
print(f"  Regionales (6 reg * 5 band) : 30")
print(f"  Left / Right (5 bandas ×2)  : 10")
print(f"  Dif. globales seleccionadas : {len(GLOBAL_DIFF_PAIRS)}")
print(f"  Dif. regionales seleccionadas: {len(REGIONAL_DIFF_PAIRS)}")
print(f"  Log-ratios (6 reg * 5 pares): {len(REGION_MAP)*len(LOG_RATIO_PAIRS)}")

  Total features por sujeto  : 183
  Raw por canal (16*5)        : 80
  Globales (5 bandas)         : 5
  Regionales (6 reg * 5 band) : 30
  Left / Right (5 bandas ×2)  : 10
  Dif. globales seleccionadas : 9
  Dif. regionales seleccionadas: 19
  Log-ratios (6 reg * 5 pares): 30


In [85]:
#Function to use per meme
def aggregate_eeg_features(eeg_raw):
    result = {"eeg_n_users": 0}
    for feat in EEG_FEATURE_NAMES:
        result[feat] = np.nan

    if not isinstance(eeg_raw, dict):
        return result

    valid_users = {u: f for u, f in eeg_raw.items() if isinstance(f, dict)}
    result["eeg_n_users"] = len(valid_users)
    if len(valid_users) == 0:
        return result

    user_rows = []
    for _, feats in valid_users.items():
        zscored = zscore_user_eeg(feats)
        user_rows.append(extract_eeg_user_features(zscored))

    user_df = pd.DataFrame(user_rows)

    for feat in EEG_FEATURE_NAMES:
        if feat in user_df.columns:
            vals = pd.to_numeric(user_df[feat], errors="coerce").dropna()
            if len(vals) > 0:
                result[feat] = float(vals.mean())

    return result

print(f"Output per meme: {len(EEG_FEATURE_NAMES) + 1} columns (features + eeg_n_users)")

Output per meme: 184 columns (features + eeg_n_users)


In [86]:
for df, name in [(df_train, "train")]:
    print(f"\nProcesando EEG — {name} ({len(df)} memes)...")
    eeg_agg = df["EEG_raw"].progress_apply(aggregate_eeg_features).apply(pd.Series)
    df[eeg_agg.columns] = eeg_agg.values
    
EEG_COLS = ["eeg_n_users"] + EEG_FEATURE_NAMES
print(f"\nTotal EEG columns: {len(EEG_COLS)}")


Procesando EEG — train (3984 memes)...


  5%|▍         | 185/3984 [00:47<16:23,  3.86it/s]


KeyboardInterrupt: 

In [ ]:
df.head()

,id,lang,text,image_file,split,ET_raw,HR_raw,EEG_raw,sexism,task21_valid,...,log_Beta_Alpha_ratio_parietal,log_Theta_Beta_ratio_parietal,log_Theta_Alpha_ratio_parietal,log_Gamma_Beta_ratio_parietal,log_Gamma_Alpha_ratio_parietal,log_Beta_Alpha_ratio_occipital,log_Theta_Beta_ratio_occipital,log_Theta_Alpha_ratio_occipital,log_Gamma_Beta_ratio_occipital,log_Gamma_Alpha_ratio_occipital
0,110887,es,A VECES QUISIERAIR/ALZUMBA www.facebook.com/Oi...,110887.jpeg,Train,"{'EN1': None, 'EN2': None, 'EN3': None, 'EN4':...","{'EN1': None, 'EN2': None, 'EN3': None, 'EN4':...","{'EN1': None, 'EN2': None, 'EN3': None, 'EN5':...",1,True,...,-0.968841,1.250080,0.269619,-0.029585,-0.195502,1.944209,NaN,-0.968583,-2.041538,-1.847240
1,110466,es,Se necesita cuidadora para adulto mayor.... fo...,110466.jpeg,Train,"{'EN1': None, 'EN2': None, 'EN3': None, 'EN4':...","{'EN1': None, 'EN2': None, 'EN3': None, 'EN4':...","{'EN1': None, 'EN2': None, 'EN3': None, 'EN5':...",1,True,...,-0.371685,0.377996,0.006312,0.191136,-0.180548,-0.498639,0.074590,-0.424049,0.410328,-0.088311
2,111269,es,tomboy como son el anime y manga pre to tomboy...,111269.jpeg,Train,"{'EN1': None, 'EN2': None, 'EN3': None, 'EN4':...","{'EN1': None, 'EN2': None, 'EN3': None, 'EN4':...","{'EN1': None, 'EN2': None, 'EN3': None, 'EN5':...",1,True,...,1.685280,0.744414,2.060350,-0.766361,1.715771,-0.044831,-0.216478,NaN,-0.376445,1.309895
3,110593,es,HOY QUIERO FELICITAR A TODAS LAS MUJERES DE ES...,110593.jpeg,Train,"{'EN1': None, 'EN2': None, 'EN3': None, 'EN4':...","{'EN1': None, 'EN2': None, 'EN3': None, 'EN4':...","{'EN1': None, 'EN2': None, 'EN3': None, 'EN5':...",0,True,...,0.073605,-0.335161,-0.027275,0.308974,0.382579,-0.941212,-1.238905,-2.180117,1.463016,0.521804
4,110946,es,DUCHATE GUARRA GRACIAS memegenerator.es,110946.jpeg,Train,"{'EN1': None, 'EN2': None, 'EN3': None, 'EN4':...","{'EN1': None, 'EN2': None, 'EN3': None, 'EN4':...","{'EN1': None, 'EN2': None, 'EN3': None, 'EN5':...",1,True,...,NaN,0.014262,0.033586,-1.655191,-0.145020,0.126780,-0.917055,NaN,-2.789226,0.549788


## 3. Labels

In [ ]:
#Re name labels for better consistency
rename_map = {
    "task21_valid_hard": "task21_valid",
    "task22_valid_hard": "task22_valid",
    "task23_valid_hard": "task23_valid",
    "task21_hard": "sexism",
    "task22_hard": "direct"}

df_train.rename(columns=rename_map, inplace=True)
#df_test.rename(columns=rename_map,  inplace=True)



In [ ]:
#Task 2.1: Sexism Detection (binary)

print("Task 2.1 — Distribución (Train):")
print("Task 2.1. Train Distribution:")
print(df_train["sexism"].value_counts().rename({0: "Non-Sexist", 1: "Sexist"}))
print(f"\nBalance: {df_train['sexism'].mean():.1%} sexists")

#if "sexism" in df_test.columns:
#    print("\nTask 2.1 — Distribución (Test):")
#    print(df_test["sexism"].value_counts().rename({0: "Non-Sexist", 1: "Sexist"}))

Task 2.1 — Distribución (Train):
Task 2.1. Train Distribution:
Sexist        2617
Non-Sexist    1367
Name: sexism, dtype: int64

Balance: 65.7% sexists


In [ ]:
#Task 2.2: Source Intention (Binary - Direct or Judgemental) 
print("Task 2.2 — Distribución (Train, solo sexistas):")
t22_train = df_train[df_train["sexism"] == 1]["direct"]
print(t22_train.value_counts().rename({1: "DIRECT", 0: "JUDGEMENTAL"}))
print(f"  NaN (no aplica): {t22_train.isna().sum()}")

#if "task22_hard" in df_test.columns:
#    print("\nTask 2.2 — Distribución (Test, solo sexistas):")
#    t22_test = df_test[df_test["task21_hard"] == 1]["task22_hard"]
#    print(t22_test.value_counts().rename({1: "DIRECT", 0: "JUDGEMENTAL"}))

Task 2.2 — Distribución (Train, solo sexistas):
DIRECT         2005
JUDGEMENTAL     612
Name: direct, dtype: int64
  NaN (no aplica): 0


In [ ]:
# Task 2.3 Sexism Categorization (Multilabel - 5 categories)
TASK23_CATEGORIES = [
    "IDEOLOGICAL-INEQUALITY",
    "STEREOTYPING-DOMINANCE",
    "OBJECTIFICATION",
    "SEXUAL-VIOLENCE",
    "MISOGYNY-NON-SEXUAL-VIOLENCE"]

TASK23_RENAME = {
    "IDEOLOGICAL-INEQUALITY":        "ideological_inequality",
    "STEREOTYPING-DOMINANCE":         "stereotyping_dominance",
    "OBJECTIFICATION":                "objectification",
    "SEXUAL-VIOLENCE":                "sexual_violence",
    "MISOGYNY-NON-SEXUAL-VIOLENCE":   "misogyny_non_sexual_violence"
}


def expand_task23(df, col, prefix):
    expanded = df[col].apply(lambda x: x if isinstance(x, dict) else {cat: np.nan for cat in TASK23_CATEGORIES}).apply(pd.Series)
    expanded.columns = [f"{prefix}{c}" for c in expanded.columns]
    return expanded


for df, name in [(df_train, "train")]:
    # Hard labels
    t23_hard = expand_task23(df, "task23_hard", "CAT_")
    df[t23_hard.columns] = t23_hard.values
    df.drop(columns=["task23_hard"], inplace=True, errors="ignore")
    print(f"Categories Expanded")

print("\nTask 2.3 Category Distribution (Train, only sexists):")
t23_cols_hard = [c for c in df_train.columns if c.startswith("CAT_")]
sexist_mask = df_train["sexism"] == 1
print(df_train.loc[sexist_mask, t23_cols_hard].sum().sort_values(ascending=False))

train: Categories Expanded

Task 2.3 Category Distribution (Train, only sexists):
CAT_STEREOTYPING-DOMINANCE          1186
CAT_OBJECTIFICATION                 1111
CAT_IDEOLOGICAL-INEQUALITY           988
CAT_SEXUAL-VIOLENCE                  539
CAT_MISOGYNY-NON-SEXUAL-VIOLENCE     433
dtype: int64


In [ ]:
fdasfd

In [ ]:
df_train.shape

## 4. Columns Selection

In [ ]:
meta_cols = ["id", "lang", "text", "image_file", "split"]

et_cols_final  = sorted([c for c in df_train.columns if c.startswith("et_")])
eeg_cols_final = sorted([c for c in df_train.columns if c.startswith("eeg_")])

#We are not going to include the HR features in the model but we will keep it for future analysis.
hr_meta_cols = [c for c in df_train.columns if c.startswith("hr_") or c == "HR_raw"]


task21_cols = ["task21_valid", "sexism"]
task22_cols = ["task22_valid", "direct"]

task23_hard_cols = sorted([c for c in df_train.columns if c.startswith("CAT_")])
task23_cols = ["task23_valid", "task23_num_hard_labels"] + task23_hard_cols 

# Columns
final_cols = (
    meta_cols +
    et_cols_final +
    eeg_cols_final +
    task21_cols +
    task22_cols +
    task23_cols)

print(f"Columnas finales: {len(final_cols)}")
print(f"  Meta:       {len(meta_cols)}")
print(f"  ET:         {len(et_cols_final)}")
print(f"  EEG:        {len(eeg_cols_final)}")
print(f"  Task21:     {len([c for c in task21_cols if c in df_train.columns])}")
print(f"  Task22:     {len([c for c in task22_cols if c in df_train.columns])}")
print(f"  Task23:     {len([c for c in task23_cols if c in df_train.columns])}")

df_train = df_train[final_cols].copy()
#df_test  = df_test[final_cols].copy()

print(f"Train final: {df_train.shape}")
#print(f"Test  final: {df_test.shape}")

df_train.head(5)

Columnas finales: 26
  Meta:       5
  ET:         9
  EEG:        1
  Task21:     2
  Task22:     2
  Task23:     7
Train final: (3984, 26)


,id,lang,text,image_file,split,et_fixation_duration_mean,et_fixation_duration_std,et_fixations_mean,et_fixations_std,et_n_users,...,sexism,task22_valid,direct,task23_valid,task23_num_hard_labels,CAT_IDEOLOGICAL-INEQUALITY,CAT_MISOGYNY-NON-SEXUAL-VIOLENCE,CAT_OBJECTIFICATION,CAT_SEXUAL-VIOLENCE,CAT_STEREOTYPING-DOMINANCE
0,110887,es,A VECES QUISIERAIR/ALZUMBA www.facebook.com/Oi...,110887.jpeg,Train,290.553894,46.333007,42.92820,19.900530,2.0,...,1,True,1.0,True,2,0,1,1,0,0
1,110466,es,Se necesita cuidadora para adulto mayor.... fo...,110466.jpeg,Train,272.211591,32.506004,48.00000,25.238859,3.0,...,1,True,1.0,True,2,0,0,0,1,1
2,111269,es,tomboy como son el anime y manga pre to tomboy...,111269.jpeg,Train,259.975251,129.980125,40.33335,3.771213,2.0,...,1,True,1.0,True,1,0,0,1,0,0
3,110593,es,HOY QUIERO FELICITAR A TODAS LAS MUJERES DE ES...,110593.jpeg,Train,303.698083,8.588806,31.90805,0.130037,2.0,...,0,True,0.0,False,0,0,0,0,0,0
4,110946,es,DUCHATE GUARRA GRACIAS memegenerator.es,110946.jpeg,Train,228.190844,132.391664,19.00000,1.414214,2.0,...,1,True,1.0,True,1,0,0,1,0,0


In [ ]:
df_train.shape

(3984, 26)

In [ ]:
#Export
train_out = os.path.join(OUTPUT_DIR, "train_model_ready.parquet")
test_out  = os.path.join(OUTPUT_DIR, "test_model_ready.parquet")

df_train.to_parquet(train_out, index=False)
#df_test.to_parquet(test_out,  index=False)

print(f"Train saved: {train_out}")
print(f"Test  saved: {test_out}")

Train saved → data\processed\train_model_ready.parquet
Test  saved → data\processed\test_model_ready.parquet
